# TimesFM Kaggle submission

Zero-shot TimesFM forecast of the **test** period (from the full sales history), written
to `submission_timesfm.csv`. Upload it to Kaggle to see how the foundation model does
on the hidden test set vs. the ensemble (3,923).

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import warnings
import numpy as np
import pandas as pd

from timesfm import ForecastConfig
from timesfm.timesfm_2p5.timesfm_2p5_torch import TimesFM_2p5_200M_torch
from src.features.nn_preprocessing import build_long_df, FREQ

warnings.filterwarnings("ignore")

train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
test_raw = pd.read_csv("data/test.csv")

# Full history (through the end of train) for each series.
Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
test_dates = pd.to_datetime(test_raw["Date"]).sort_values().unique()
horizon_test = len(test_dates)
print("test weeks (horizon):", horizon_test)

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


test weeks (horizon): 39


In [3]:
model = TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(ForecastConfig(
    max_context=512,
    max_horizon=horizon_test,
    normalize_inputs=True,
    infer_is_positive=True,
    per_core_batch_size=32,
))
print("TimesFM ready.")

TimesFM ready.


In [4]:
uids, series_list, last_ds = [], [], {}
for uid, g in Y_df.groupby("unique_id", sort=False):
    g = g.sort_values("ds")
    uids.append(uid)
    series_list.append(g["y"].values.astype(float))
    last_ds[uid] = g["ds"].max()

point_forecast, _ = model.forecast(horizon=horizon_test, inputs=series_list)
point_forecast = np.asarray(point_forecast)

rows = []
for i, uid in enumerate(uids):
    dates = pd.date_range(last_ds[uid] + pd.Timedelta(weeks=1), periods=horizon_test, freq=FREQ)
    for h in range(horizon_test):
        rows.append((uid, dates[h], float(point_forecast[i][h])))
fcst = pd.DataFrame(rows, columns=["unique_id", "ds", "timesfm"])
print("forecast rows:", len(fcst))

forecast rows: 129909


In [5]:
test_frame = test_raw[["Store", "Dept", "Date"]].copy()
test_frame["unique_id"] = test_frame["Store"].astype(str) + "_" + test_frame["Dept"].astype(str)
test_frame["ds"] = pd.to_datetime(test_frame["Date"])

out = test_frame.merge(fcst, on=["unique_id", "ds"], how="left")
out["Weekly_Sales"] = out["timesfm"].fillna(0).clip(lower=0)
out["Id"] = out["Store"].astype(str) + "_" + out["Dept"].astype(str) + "_" + out["Date"].astype(str)

submission = out[["Id", "Weekly_Sales"]]
submission.to_csv("submission_timesfm.csv", index=False)
print("rows:", len(submission), "| NaNs:", int(submission["Weekly_Sales"].isna().sum()),
      "| matched:", int(out["timesfm"].notna().sum()))
submission.head()

rows: 115064 | NaNs: 0 | matched: 114661


,Id,Weekly_Sales
0,1_1_2012-11-02,29630.828125
1,1_1_2012-11-09,24433.855469
2,1_1_2012-11-16,21384.291016
3,1_1_2012-11-23,21034.988281
4,1_1_2012-11-30,25151.812500
